# 항만 대회: 공개 LLM으로 train 선박명 한글 표기 증강

- 대회 공지(talkboard 418501, 10/23): train 데이터를 이용한 LLM 증강은 허용. 생성 데이터와 생성 코드는 보관.
- 입력: train.csv에 나온 선박 5,851척의 영문 카탈로그명 (`llm_aug_input.csv`)
- 예시: train의 실제 한글-영문 짝 30개 (`llm_fewshot.csv`)
- 모델: Qwen2.5-7B-Instruct (Apache 2.0). A100/L4는 vLLM, T4는 transformers 4비트로 자동 선택
- 출력: `llm_korean_aug.csv` (vessel_id, en_name, ko_greedy, ko_sample1, ko_sample2)

**런타임 → 런타임 유형 변경 → GPU** 로 바꾼 뒤 위에서부터 차례로 실행하세요. (A100 또는 L4면 약 10분, T4면 약 1시간)

In [ ]:
import torch, subprocess
cc = torch.cuda.get_device_capability()
print(torch.cuda.get_device_name(), cc)
USE_VLLM = cc[0] >= 8   # vLLM needs Ampere or newer; T4 (7.5) uses transformers
pkgs = 'vllm' if USE_VLLM else 'bitsandbytes accelerate'
print(subprocess.run(f'pip -q install {pkgs}', shell=True, capture_output=True, text=True).stdout[-500:])
print('USE_VLLM =', USE_VLLM)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/항만_llm_aug'
print(os.listdir(BASE))

In [ ]:
import re, time
import pandas as pd

inp = pd.read_csv(f'{BASE}/llm_aug_input.csv')
fs = pd.read_csv(f'{BASE}/llm_fewshot.csv')
examples = '\n'.join(f'English: {e}\nKorean: {k}' for e, k in zip(fs.en_name, fs.ko_name))
SYSTEM = ('You write the Hangul name that a Korean port authority would record for a foreign or Korean ship. '
          'Transliterate by pronunciation (외래어 표기); do not translate meanings. '
          'Letters of an acronym are read as Korean letter names (MS -> 엠에스, KMTC -> 케이엠티씨). '
          '"No. N" becomes 제N. Answer with the Hangul name only.\n\nExamples:\n' + examples)

def messages(e):
    return [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': f'English: {e}\nKorean:'}]

def clean(s):
    s = s.strip().split('\n')[0]
    s = re.sub(r'^(Korean|한글)\s*[:：]\s*', '', s)
    s = re.sub(r'[^0-9A-Za-z가-힣 ]', ' ', s)
    return ' '.join(s.split())

In [ ]:
t = time.time()
if USE_VLLM:
    from vllm import LLM, SamplingParams
    llm = LLM(model='Qwen/Qwen2.5-7B-Instruct', dtype='bfloat16', max_model_len=2048,
              enable_prefix_caching=True, gpu_memory_utilization=0.90, seed=0)
    tok = llm.get_tokenizer()
    prompts = [tok.apply_chat_template(messages(e), tokenize=False, add_generation_prompt=True) for e in inp.en_name]
    greedy = [o.outputs[0].text for o in llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=32, stop=['\n']))]
    samp = llm.generate(prompts, SamplingParams(n=2, temperature=0.8, top_p=0.95, max_tokens=32, stop=['\n'], seed=1234))
    s1 = [o.outputs[0].text for o in samp]; s2 = [o.outputs[1].text for o in samp]
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    name = 'Qwen/Qwen2.5-7B-Instruct'
    tok = AutoTokenizer.from_pretrained(name); tok.padding_side = 'left'
    model = AutoModelForCausalLM.from_pretrained(name, device_map='cuda', quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')).eval()
    prompts = [tok.apply_chat_template(messages(e), tokenize=False, add_generation_prompt=True) for e in inp.en_name]
    def run(sample, seed):
        torch.manual_seed(seed); res = []
        for i in range(0, len(prompts), 48):
            enc = tok(prompts[i:i + 48], return_tensors='pt', padding=True).to('cuda')
            with torch.no_grad():
                g = model.generate(**enc, max_new_tokens=24, do_sample=sample, temperature=0.8 if sample else None,
                                   top_p=0.95 if sample else None, pad_token_id=tok.eos_token_id)
            res += tok.batch_decode(g[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
            if i % 960 == 0: print(i, f'{time.time() - t:.0f}s', flush=True)
        return res
    greedy = run(False, 0); s1 = run(True, 1234); s2 = run(True, 5678)
print(f'{time.time() - t:.0f}s')

out = inp.copy()
out['ko_greedy'] = [clean(x) for x in greedy]
out['ko_sample1'] = [clean(x) for x in s1]
out['ko_sample2'] = [clean(x) for x in s2]
out.to_csv(f'{BASE}/llm_korean_aug.csv', index=False, encoding='utf-8-sig')
print(out.sample(15, random_state=0).to_string())

In [ ]:
# 품질 확인: train에 실제 한글 표기가 있는 선박과 비교 (예시 30척 제외)
from difflib import SequenceMatcher
ref = pd.read_csv(f'{BASE}/train_korean_ref.csv')
m = ref[~ref.vessel_id.isin(fs.vessel_id)].merge(out, on='vessel_id')
m['sim'] = [SequenceMatcher(None, a.replace(' ', ''), b.replace(' ', '')).ratio() for a, b in zip(m.ko_name, m.ko_greedy)]
print(len(m), 'vessels | exact', (m.ko_name.str.replace(' ', '') == m.ko_greedy.str.replace(' ', '')).mean().round(3),
      '| mean char similarity', m.sim.mean().round(3))
print(m.sort_values('sim').head(10)[['en_name', 'ko_name', 'ko_greedy', 'sim']].to_string())